In [1]:
ZIP = "analysis-export.zip"
RUN = 2

In [2]:
import io, json, re, zipfile
from itertools import combinations
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score, precision_recall_fscore_support
import krippendorff
from irrCAC.raw import CAC

### Carregamento de dados

In [3]:
REFERENCE_RUN = RUN

# a exportacao Go escreve CSVs em snake_case; renomeia-se para o camelCase que as funcoes usam.
RENAME = {"individual_id":"individualId","external_game_id":"externalGameId","model_slug":"modelSlug",
          "final_label":"finalLabel","finish_reason":"finishReason","run_id":"runId",
          "completion_tokens":"completionTokens","raw_response":"rawResponse","game_name":"gameName",
          "population_id":"populationId","taxonomy_version":"taxonomyVersion","game_context":"gameContext",
          "selection_prob":"selectionProb","panel_run_id":"panelRunId","sample_id":"sampleId"}

In [4]:
def open_zip(filename=None):
    """funciona no Colab (pedido de carregamento) e localmente (ficheiro na directoria de trabalho)."""
    try:
        from google.colab import files
        up = files.upload()
        return zipfile.ZipFile(io.BytesIO(next(iter(up.values()))))
    except ImportError:
        return zipfile.ZipFile(filename or ZIP)

In [5]:
_zf = None
def _zip():
    """abre o zip na primeira utilizacao (e nao no import), para que importar shared nunca falhe."""
    global _zf
    if _zf is None:
        _zf = open_zip()
    return _zf


In [6]:
_cache = {}
def load(name, run=REFERENCE_RUN):
    """le um CSV do zip, com cache por (name, run); run=None devolve todas as execucoes.
    O exportador escreve os booleanos como strings "true"/"false"; reconvertem-se aqui."""
    key = (name, run)
    if key not in _cache:
        df = pd.read_csv(_zip().open(name)).rename(columns=RENAME)
        if "runId" in df.columns and run is not None:
            df = df[df["runId"] == run]
        for col in ("present", "finalLabel"):
            if col in df.columns:
                df = df.assign(**{col: df[col].astype(str).str.lower().eq("true")})
        _cache[key] = df
    return _cache[key].copy()

### Painel e regras de agregação

In [7]:
def panel_majority(ann):
    """reduz o painel a um unico avaliador 'panel-majority': presente onde uma maioria estrita sinalizou."""
    g = ann.groupby(["individualId", "code"])["present"]
    maj = (g.sum() * 2 > g.count()).reset_index()
    maj["modelSlug"] = "panel-majority"
    return maj

In [8]:
def panel_vote(ann, k, label):
    """agrega o painel sob uma regra mais permissiva/exigente: presente onde >= k dos modelos que sinalizaram.
    k=1 é a uniao ('qualquer modelo'), k=3 é a maioria estrita de quatro, k=4 é a unanimidade."""
    g = ann.groupby(["individualId", "code"])["present"]
    out = (g.sum() >= k).reset_index()
    out["modelSlug"] = label
    return out

### Concordância (fiabilidade)

In [9]:
def _matrix(ann, code):
    sub = ann[ann["code"] == code]
    return sub.pivot_table(index="modelSlug", columns="individualId", values="present", aggfunc="first")

In [10]:
def alpha(ann, code):
    m = _matrix(ann, code).astype(float).values
    try:    return krippendorff.alpha(reliability_data=m, level_of_measurement="nominal")
    except Exception: return np.nan  # sem variancia (todos disseram ausente) -> indefinido


In [11]:
def pairwise_kappa(ann, code):
    m = _matrix(ann, code); ks = []
    for a, b in combinations(m.index, 2):
        pair = m.loc[[a, b]].dropna(axis=1)
        if pair.shape[1] < 2: continue
        x, y = pair.iloc[0].astype(int), pair.iloc[1].astype(int)
        ks.append(1.0 if (x.nunique()==1 and y.nunique()==1 and x.iloc[0]==y.iloc[0]) else cohen_kappa_score(x, y))
    return float(np.nanmean(ks)) if ks else np.nan

In [12]:
def ac1(ann, code):
    m = _matrix(ann, code).T
    try:    return CAC(m.astype(float)).gwet()["est"]["coefficient_value"]
    except Exception: return np.nan

In [13]:
def specific_agreement(ann, code):
    m = _matrix(ann, code); pp=pn=pd_=nd=0
    for u in m.columns:
        col = m[u].dropna(); n = len(col)
        if n < 2: continue
        x = int(col.sum()); y = n - x
        pp += x*(x-1); pn += y*(y-1); pd_ += x*(n-1); nd += y*(n-1)
    return (pp/pd_ if pd_ else np.nan), (pn/nd if nd else np.nan)

In [14]:
def agreement_table(ann):
    rows = []
    for code in sorted(ann["code"].unique()):
        p_pos, p_neg = specific_agreement(ann, code)
        rows.append({"code": code, "alpha": alpha(ann, code), "kappa": pairwise_kappa(ann, code),
                     "ac1": ac1(ann, code), "p_pos": p_pos, "p_neg": p_neg,
                     "pos_votes": int(np.nan_to_num(_matrix(ann, code).astype(float).to_numpy()).sum())})
    return pd.DataFrame(rows).round(3)

### Calibração e direcção (validade)

In [15]:
def cells(ann, gold, pass_="open"):
    """uma linha por (modelo, padrao, revisao) com rotulo gold nesta passagem: y_true (gold) / y_pred (modelo)."""
    g = gold[gold["pass"] == pass_][["individualId", "code", "finalLabel"]]
    full = pd.concat([ann[["individualId","code","modelSlug","present"]], panel_majority(ann)], ignore_index=True)
    df = full.merge(g, on=["individualId","code"], how="inner").rename(columns={"modelSlug":"model"})
    df["y_true"] = df["finalLabel"].astype(int); df["y_pred"] = df["present"].astype(int)
    return df[["model","code","individualId","y_true","y_pred"]]

In [16]:
def calibration(cell_df):
    rows = []
    for (model, code), d in cell_df.groupby(["model","code"]):
        p, r, f, _ = precision_recall_fscore_support(d["y_true"], d["y_pred"], average="binary", pos_label=1, zero_division=0)
        rows.append({"model":model,"code":code,"precision":round(p,3),"recall":round(r,3),"f1":round(f,3),"support":int(d["y_true"].sum())})
    return pd.DataFrame(rows)

In [17]:
def macro(cal):
    return cal.groupby("model")[["precision","recall","f1"]].mean().round(3)

In [18]:
def bootstrap_macro_f1(cell_df, n=2000, seed=0):
    """IC a 95% do F1 macro de cada modelo, reamostrando revisoes com reposicao."""
    rng = np.random.default_rng(seed); ids = cell_df["individualId"].unique()
    samples = {m: [] for m in cell_df["model"].unique()}
    for _ in range(n):
        take = pd.DataFrame({"individualId": rng.choice(ids, len(ids), replace=True)})
        f1 = macro(calibration(take.merge(cell_df, on="individualId", how="left")))["f1"]
        for m in samples: samples[m].append(f1.get(m, np.nan))
    return {m: (round(np.nanpercentile(v,2.5),3), round(np.nanpercentile(v,97.5),3)) for m, v in samples.items()}

In [19]:
def adjudication_direction(ann, gold, pass_="open"):
    g = gold[gold["pass"] == pass_][["individualId","code","finalLabel"]]
    maj = panel_majority(ann)[["individualId","code","present"]].rename(columns={"present":"maj"})
    m = g.merge(maj, on=["individualId","code"], how="inner")
    m["dir"] = np.where(m["finalLabel"] == m["maj"], "confirmation", "replacement")
    t = m.groupby("code")["dir"].value_counts().unstack(fill_value=0)
    for c in ("confirmation","replacement"):
        if c not in t: t[c] = 0
    t["override_rate"] = (t["replacement"] / (t["confirmation"] + t["replacement"])).round(3)
    indep = int(((m["maj"] == False) & (m["finalLabel"] == True)).sum())
    print(f"adicoes independentes (painel ausente, autor presente): {indep} / {len(m)} celulas")
    return t[["confirmation","replacement","override_rate"]].sort_values("override_rate", ascending=False)

### Saúde e modos de falha

In [20]:
FENCE = re.compile(r"```")
COMMENT = re.compile(r"(^|\n)\s*//|/\*")
def classify(text):
    """classifica uma resposta falhada pelo seu defeito de forma (a mensagem do json.loads decide)."""
    text = str(text)
    if not text.strip():           return "empty"
    if FENCE.search(text):         return "markdown_fence"
    if COMMENT.search(text):       return "js_comment"
    try:
        json.loads(text);          return "schema_rejected"
    except json.JSONDecodeError as e:
        m = e.msg.lower()
        if "unterminated string" in m: return "unescaped_string"
        if "delimiter" in m:           return "missing_delimiter"
        if "expecting value" in m:     return "expecting_value"
        if "extra data" in m:          return "trailing_data"
        return "other_syntax"

In [21]:
def status_by_model(status):
    g = status.groupby("modelSlug")["status"]
    out = pd.DataFrame({"parse_err": g.apply(lambda s: int((s != "completed").sum())), "total": g.size()})
    out["pct"] = (100 * out["parse_err"] / out["total"]).round(2)
    return out.sort_values("pct", ascending=False)

In [22]:
def partial_panel(status):
    bad = status.loc[status["status"] != "completed", "individualId"].nunique()
    tot = status["individualId"].nunique()
    return bad, tot, round(100 * bad / tot, 1) if tot else 0.0

In [23]:
def adjudicated_on_partial(status, gold, pass_="open"):
    adj = set(gold.loc[gold["pass"] == pass_, "individualId"])
    bad = set(status.loc[status["status"] != "completed", "individualId"])
    return len(adj & bad), len(adj), round(100 * len(adj & bad) / len(adj), 1) if adj else 0.0

### Estilo de figuras

In [24]:
INK, MID, FAINT = "#1c1c1e", "#55555c", "#c9c9cf"

In [25]:
def styled_plt():
    """devolve o pyplot ja configurado: tons de cinzento, sem grelha pesada."""
    import matplotlib.pyplot as plt
    plt.rcParams.update({
        "figure.dpi": 110, "savefig.dpi": 200,
        "font.size": 9.5, "axes.labelsize": 9.5,
        "axes.edgecolor": MID, "axes.linewidth": 0.8,
        "axes.spines.top": False, "axes.spines.right": False,
        "xtick.color": MID, "ytick.color": MID,
        "text.color": INK, "axes.labelcolor": INK,
        "legend.frameon": False,
    })
    return plt

In [26]:
def virgula(x, _pos=None):
    """rotulo numerico com virgula decimal (0.45 -> 0,45)."""
    return f"{x:g}".replace(".", ",")

In [27]:
def save_fig(fig, name):
    """grava a figura em figures/<name>.png junto do notebook (dpi 200, corte justo)."""
    import os
    os.makedirs("figures", exist_ok=True)
    path = os.path.join("figures", name + ".png")
    fig.savefig(path, bbox_inches="tight")
    print("gravado:", path)